# Char-Level Transformer Transliteration

This notebook keeps the same Nepali transliteration data pipeline but replaces the GRU-based seq2seq model with a lightweight encoder-decoder Transformer.

In [14]:
import json
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except ImportError:
    class _TqdmFallback:
        def __init__(self, iterable=None, **kwargs):
            self.iterable = iterable if iterable is not None else []

        def __iter__(self):
            return iter(self.iterable)

        def set_postfix(self, *args, **kwargs):
            return None

        def close(self):
            return None

    def tqdm(iterable=None, **kwargs):
        return _TqdmFallback(iterable, **kwargs)

In [15]:
def resolve_data_file(*candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path

    kaggle_root = Path('/kaggle/input')
    if kaggle_root.exists():
        for candidate in candidates:
            matches = sorted(kaggle_root.rglob(Path(candidate).name))
            if matches:
                return matches[0]

    raise FileNotFoundError(f'Could not locate any of: {candidates}')


def read_json_records(path):
    text = Path(path).read_text(encoding='utf-8')
    stripped = text.lstrip()
    if stripped.startswith('['):
        return json.loads(text)
    return [json.loads(line) for line in text.splitlines() if line.strip()]


def normalize_text(text):
    text = unicodedata.normalize('NFKC', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


train_path = resolve_data_file('nep_train.json')
valid_path = resolve_data_file('nep_ valid.json', 'nep_valid.json')
test_path = resolve_data_file('nep_test.json')

train_records = read_json_records(train_path)
dev_records = read_json_records(valid_path)
test_records = read_json_records(test_path)

for dataset in (train_records, dev_records, test_records):
    for record in dataset:
        record['english word'] = normalize_text(record['english word']).lower()
        record['native word'] = normalize_text(record['native word'])

random.Random(42).shuffle(train_records)
MAX_TRAIN_EXAMPLES = 300_000
if len(train_records) > MAX_TRAIN_EXAMPLES:
    train_records = train_records[:MAX_TRAIN_EXAMPLES]

print('Train size:', len(train_records))
print('Valid size:', len(dev_records))
print('Test size:', len(test_records))

Train size: 300000
Valid size: 2804
Test size: 4101


In [16]:
SPECIAL_TOKENS = ['<pad>', '<sos>', '<eos>', '<unk>']

def build_char_vocab(records):
    chars = Counter()
    for record in records:
        chars.update(record['english word'])
        chars.update(record['native word'])
    itos = SPECIAL_TOKENS + sorted(chars)
    stoi = {token: idx for idx, token in enumerate(itos)}
    return stoi, itos

char_to_id, id_to_char = build_char_vocab(train_records)
PAD_ID = char_to_id['<pad>']
SOS_ID = char_to_id['<sos>']
EOS_ID = char_to_id['<eos>']
UNK_ID = char_to_id['<unk>']
VOCAB_SIZE = len(id_to_char)

def estimate_max_length(records, field, cap=48, percentile=0.98):
    lengths = [len(normalize_text(record[field])) for record in records]
    if not lengths:
        return cap
    sorted_lengths = sorted(lengths)
    index = int(round((len(sorted_lengths) - 1) * percentile))
    index = max(0, min(index, len(sorted_lengths) - 1))
    return min(cap, sorted_lengths[index] + 2)

SRC_MAX_LEN = estimate_max_length(train_records, 'english word', cap=48)
TGT_MAX_LEN = estimate_max_length(train_records, 'native word', cap=48)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def text_to_ids(text, add_sos=True, add_eos=True, max_length=None):
    text = normalize_text(text)
    ids = []
    if add_sos:
        ids.append(SOS_ID)
    for char in text:
        ids.append(char_to_id.get(char, UNK_ID))
        if max_length is not None and len(ids) >= max_length - int(add_eos):
            break
    if add_eos:
        ids.append(EOS_ID)
    if max_length is not None:
        ids = ids[:max_length]
    return ids

def ids_to_text(ids):
    chars = []
    for idx in ids:
        token = id_to_char[int(idx)]
        if token == '<eos>':
            break
        if token in SPECIAL_TOKENS:
            continue
        chars.append(token)
    return ''.join(chars)

print(f'Vocab size: {VOCAB_SIZE}')
print(f'Source max length: {SRC_MAX_LEN}')
print(f'Target max length: {TGT_MAX_LEN}')
print(f'Device: {DEVICE}')

Vocab size: 93
Source max length: 21
Target max length: 19
Device: cuda


In [17]:
import pickle

# Save char_to_id, id_to_char, SRC_MAX_LEN, TGT_MAX_LEN in a single file for easy reuse
vocab_info = {
    "char_to_id": char_to_id,
    "id_to_char": id_to_char,
    "PAD_ID": PAD_ID,
    "SOS_ID": SOS_ID,
    "EOS_ID": EOS_ID,
    "UNK_ID": UNK_ID,
    "SRC_MAX_LEN": SRC_MAX_LEN,
    "TGT_MAX_LEN": TGT_MAX_LEN,
    "VOCAB_SIZE": VOCAB_SIZE,
    "SPECIAL_TOKENS": SPECIAL_TOKENS
}

with open("char_vocab.pkl", "wb") as f:
    pickle.dump(vocab_info, f)

print("Saved vocabulary and mappings to char_vocab.pkl")

Saved vocabulary and mappings to char_vocab.pkl


In [5]:
def delete_char(word, prob=0.08):
    if len(word) > 3 and random.random() < prob:
        idx = random.randrange(len(word))
        return word[:idx] + word[idx + 1 :]
    return word


def duplicate_char(word, prob=0.08):
    if len(word) > 2 and random.random() < prob:
        idx = random.randrange(len(word))
        return word[: idx + 1] + word[idx] + word[idx + 1 :]
    return word


def swap_char(word, prob=0.08):
    if len(word) > 3 and random.random() < prob:
        idx = random.randrange(len(word) - 1)
        return word[:idx] + word[idx + 1] + word[idx] + word[idx + 2 :]
    return word


PHONETIC_RULES = [
    ("chh", ["ch", "x"]),
    ("ch", ["c", "x"]),
    ("kh", ["k"]),
    ("gh", ["g"]),
    ("ph", ["f", "p"]),
    ("bh", ["b", "v"]),
    ("th", ["t"]),
    ("dh", ["d"]),
    ("sh", ["s"]),
    ("aa", ["a"]),
    ("ee", ["i"]),
    ("oo", ["u"]),
    ("ai", ["e"]),
    ("au", ["o"]),
]

TYPO_REPLACEMENTS = {
    "a": ["e", "o"],
    "b": ["p"],
    "c": ["k", "s"],
    "d": ["t"],
    "e": ["i", "a"],
    "g": ["k"],
    "i": ["e", "y"],
    "k": ["c", "q"],
    "l": ["r"],
    "m": ["n"],
    "n": ["m"],
    "o": ["u", "a"],
    "p": ["b"],
    "r": ["l"],
    "s": ["z"],
    "t": ["d"],
    "u": ["o"],
    "v": ["b"],
    "x": ["ch"],
    "y": ["i"],
    "z": ["s"],
}

INSERTABLE_CHARS = 'abcdefghijklmnopqrstuvwxyz'


def phonetic_noise(text, prob=0.18):
    for key, replacements in PHONETIC_RULES:
        if key in text and random.random() < prob:
            text = text.replace(key, random.choice(replacements), 1)
    return text


def insert_char(text, prob=0.06):
    if len(text) > 1 and random.random() < prob:
        idx = random.randrange(len(text) + 1)
        return text[:idx] + random.choice(INSERTABLE_CHARS) + text[idx:]
    return text


def transpose_far_chars(text, prob=0.05):
    if len(text) > 4 and random.random() < prob:
        left, right = sorted(random.sample(range(len(text)), 2))
        chars = list(text)
        chars[left], chars[right] = chars[right], chars[left]
        return ''.join(chars)
    return text


def visual_substitution(text, prob=0.10):
    if len(text) == 0 or random.random() >= prob:
        return text
    indexes = [idx for idx, char in enumerate(text) if char in TYPO_REPLACEMENTS]
    if not indexes:
        return text
    idx = random.choice(indexes)
    replacement = random.choice(TYPO_REPLACEMENTS[text[idx]])
    return text[:idx] + replacement + text[idx + 1 :]


def merge_space_noise(text, prob=0.08):
    if ' ' not in text or random.random() >= prob:
        return text
    parts = text.split()
    if len(parts) < 2:
        return text
    index = random.randrange(len(parts) - 1)
    parts[index] = parts[index] + parts[index + 1]
    del parts[index + 1]
    return ' '.join(parts)


def apply_typo_noise(text, num_ops=1):
    operations = [
        delete_char,
        duplicate_char,
        swap_char,
        insert_char,
        transpose_far_chars,
        visual_substitution,
        merge_space_noise,
    ]
    for _ in range(num_ops):
        text = random.choice(operations)(text)
    return text


NOISE_STATE = {
    'apply_prob': 0.35,
    'heavy_prob': 0.25,
    'light_ops': (1, 2),
    'heavy_ops': (3, 4),
    'phonetic_prob': 0.18,
    'phonetic_layer_prob': 0.70,
    'typo_layer_prob': 0.90,
}


def set_noise_schedule(epoch, total_epochs):
    progress = 0.0 if total_epochs <= 1 else (epoch - 1) / (total_epochs - 1)
    NOISE_STATE['apply_prob'] = 0.20 + 0.15 * progress
    NOISE_STATE['heavy_prob'] = 0.10 + 0.35 * progress
    NOISE_STATE['phonetic_prob'] = 0.12 + 0.08 * progress
    NOISE_STATE['phonetic_layer_prob'] = 0.55 + 0.20 * progress
    NOISE_STATE['typo_layer_prob'] = 0.80 + 0.10 * progress


def add_noise(text, apply_prob=None, profile='mixed'):
    text = normalize_text(text).lower()
    config = NOISE_STATE.copy()

    if apply_prob is not None:
        config['apply_prob'] = apply_prob

    if profile == 'heavy':
        config['apply_prob'] = 1.0
        config['heavy_prob'] = 1.0
        config['phonetic_layer_prob'] = 1.0
        config['typo_layer_prob'] = 1.0
        config['phonetic_prob'] = max(config['phonetic_prob'], 0.22)
    elif profile == 'light':
        config['apply_prob'] = min(config['apply_prob'], 0.25)
        config['heavy_prob'] = 0.0
        config['phonetic_layer_prob'] = min(config['phonetic_layer_prob'], 0.50)
        config['typo_layer_prob'] = min(config['typo_layer_prob'], 0.75)

    if random.random() > config['apply_prob']:
        return text

    if random.random() < config['phonetic_layer_prob']:
        text = phonetic_noise(text, prob=config['phonetic_prob'])

    if random.random() < config['typo_layer_prob']:
        if profile == 'heavy' or random.random() < config['heavy_prob']:
            num_operations = random.randint(*config['heavy_ops'])
        else:
            num_operations = random.randint(*config['light_ops'])
        text = apply_typo_noise(text, num_ops=num_operations)

    if random.random() < 0.25:
        text = phonetic_noise(text, prob=config['phonetic_prob'] / 2)

    return re.sub(r'\s+', ' ', text).strip()


In [ ]:
class TransliterationDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        src = record['english word']
        tgt = record['native word']

        if self.augment:
            src = add_noise(src)

        src_ids = torch.tensor(text_to_ids(src, max_length=SRC_MAX_LEN), dtype=torch.long)
        tgt_ids = torch.tensor(text_to_ids(tgt, max_length=TGT_MAX_LEN), dtype=torch.long)
        return {'src_ids': src_ids, 'tgt_ids': tgt_ids}

In [ ]:
def collate_batch(batch):
    src_sequences = [item['src_ids'] for item in batch]
    tgt_sequences = [item['tgt_ids'] for item in batch]
    src_lengths = torch.tensor([len(sequence) for sequence in src_sequences], dtype=torch.long)
    tgt_lengths = torch.tensor([len(sequence) for sequence in tgt_sequences], dtype=torch.long)
    src_batch = pad_sequence(src_sequences, batch_first=True, padding_value=PAD_ID)
    tgt_batch = pad_sequence(tgt_sequences, batch_first=True, padding_value=PAD_ID)
    return {'src': src_batch, 'src_lengths': src_lengths, 'tgt': tgt_batch, 'tgt_lengths': tgt_lengths}

batch_size = 64 if torch.cuda.is_available() else 16
train_dataset = TransliterationDataset(train_records, augment=True)
dev_dataset = TransliterationDataset(dev_records, augment=False)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=torch.cuda.is_available(), collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available(), collate_fn=collate_batch)

print(f'Train batches: {len(train_loader)}')
print(f'Valid batches: {len(dev_loader)}')

Train batches: 4688
Valid batches: 44


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=256):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2, dtype=torch.float32) * (-torch.log(torch.tensor(10000.0)) / embed_dim))
        pe = torch.zeros(max_len, embed_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

In [ ]:
class CharTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, num_heads=8, ff_dim=512, num_encoder_layers=3, num_decoder_layers=3, dropout=0.15):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.positional_encoding = PositionalEncoding(embed_dim, dropout=dropout, max_len=max(SRC_MAX_LEN, TGT_MAX_LEN) + 16)
        self.transformer = nn.Transformer(
            d_model=embed_dim,
            nhead=num_heads,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
        )
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def make_padding_mask(self, tokens):
        return tokens == PAD_ID

    def make_causal_mask(self, size, device):
        return torch.triu(torch.ones(size, size, device=device, dtype=torch.bool), diagonal=1)

    def encode(self, src):
        src_emb = self.positional_encoding(self.embedding(src))
        src_padding_mask = self.make_padding_mask(src)
        return src_emb, src_padding_mask

    def decode(self, tgt, memory, tgt_padding_mask, memory_padding_mask):
        tgt_emb = self.positional_encoding(self.embedding(tgt))
        tgt_mask = self.make_causal_mask(tgt.size(1), tgt.device)
        decoded = self.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=memory_padding_mask,
        )
        return self.output_layer(decoded)

    def forward(self, src, src_lengths=None, trg=None, max_len=TGT_MAX_LEN):
        src_emb, src_padding_mask = self.encode(src)
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

        if trg is not None:
            decoder_input = trg[:, :-1]
            decoder_padding_mask = decoder_input == PAD_ID
            logits = self.decode(decoder_input, memory, decoder_padding_mask, src_padding_mask)
            return logits

        batch_size = src.size(0)
        generated = torch.full((batch_size, 1), SOS_ID, dtype=torch.long, device=src.device)
        for _ in range(max_len - 1):
            decoder_padding_mask = generated == PAD_ID
            logits = self.decode(generated, memory, decoder_padding_mask, src_padding_mask)
            next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)
            if torch.all(next_token.squeeze(1) == EOS_ID):
                break
        return generated

In [8]:
def levenshtein_distance(left, right):
    if len(left) < len(right):
        left, right = right, left
    previous_row = list(range(len(right) + 1))
    for i, left_char in enumerate(left, start=1):
        current_row = [i]
        for j, right_char in enumerate(right, start=1):
            insert_cost = current_row[j - 1] + 1
            delete_cost = previous_row[j] + 1
            replace_cost = previous_row[j - 1] + (left_char != right_char)
            current_row.append(min(insert_cost, delete_cost, replace_cost))
        previous_row = current_row
    return previous_row[-1]


def evaluate_model(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    total_distance = 0
    total_characters = 0
    total_exact_match = 0

    with torch.no_grad():
        for batch in loader:
            src = batch['src'].to(DEVICE)
            tgt = batch['tgt'].to(DEVICE)
            outputs = model(src, trg=tgt)
            loss = criterion(outputs.reshape(-1, VOCAB_SIZE), tgt[:, 1:].reshape(-1))
            total_loss += loss.item() * src.size(0)
            predictions = model(src, trg=None, max_len=TGT_MAX_LEN)
            for prediction, target in zip(predictions, tgt):
                predicted_text = ids_to_text(prediction.tolist())
                target_text = ids_to_text(target.tolist())
                total_distance += levenshtein_distance(predicted_text, target_text)
                total_characters += max(len(target_text), 1)
                total_exact_match += int(predicted_text == target_text)
                total_examples += 1

    return (
        total_loss / max(total_examples, 1),
        total_distance / max(total_characters, 1),
        total_exact_match / max(total_examples, 1),
    )

In [9]:
model = CharTransformer(VOCAB_SIZE, embed_dim=256, num_heads=8, ff_dim=512, num_encoder_layers=3, num_decoder_layers=3, dropout=0.15).to(DEVICE)
print(model)
print(f'Trainable parameters: {sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad):,}')

CharTransformer(
  (embedding): Embedding(93, 256, padding_idx=0)
  (positional_encoding): PositionalEncoding(
    (dropout): Dropout(p=0.15, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.15, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.15, inplace=False)
          (dropout2): Dropout(p=0.15, inplace=False)
        )
      )
      (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    )
 

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
learning_rate = 5e-4
weight_decay = 1e-4
grad_clip = 1.0
num_epochs = 120
patience = 10
grad_accumulation_steps = 2 if torch.cuda.is_available() else 1
use_amp = torch.cuda.is_available()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

print(f'AMP enabled: {use_amp}')
print(f'Learning rate: {learning_rate}')
print(f'Epochs: {num_epochs}')
print(f'Gradient accumulation steps: {grad_accumulation_steps}')

AMP enabled: True
Learning rate: 0.0005
Epochs: 120
Gradient accumulation steps: 2


In [ ]:
def model_has_nonfinite(model):
    """Returns True if any parameter is non-finite (NaN or Inf)."""
    for name, param in model.named_parameters():
        if not torch.isfinite(param.data).all():
            print(f"[Non-finite weights detected in: {name}]")
            return True
    return False

def clamp_noise_state():
    # Assumes global or module-level NOISE_STATE dict
    NOISE_STATE["apply_prob"] = min(NOISE_STATE.get("apply_prob", 0.2), 0.33)
    NOISE_STATE["heavy_prob"] = min(NOISE_STATE.get("heavy_prob", 0.1), 0.18)

checkpoint_path = Path('char_transformer_best.pt')
best_val_cer = float('inf')
patience_counter = 0
history = []

start_epoch = 1
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    scheduler.load_state_dict(checkpoint['scheduler_state'])
    scaler.load_state_dict(checkpoint['scaler_state'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_cer = checkpoint['best_val_cer']
    patience_counter = checkpoint.get('patience_counter', 0)
    history = checkpoint.get('history', [])
    print(f"Resumed from epoch {checkpoint['epoch']} (start_epoch={start_epoch}), best_val_cer={best_val_cer:.4f}")

exploded_loss_threshold = 3.0 # sensible initial value

for epoch in range(start_epoch, num_epochs + 1):
    set_noise_schedule(epoch, num_epochs)
    clamp_noise_state()  # <- to prevent runaway noise
    model.train()
    running_loss = 0.0
    total_examples = 0
    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=False, dynamic_ncols=True)

    for step, batch in enumerate(progress_bar, start=1):
        src = batch['src'].to(DEVICE)
        tgt = batch['tgt'].to(DEVICE)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
            outputs = model(src, trg=tgt)
            loss = criterion(outputs.reshape(-1, VOCAB_SIZE), tgt[:, 1:].reshape(-1))
            loss = loss / grad_accumulation_steps

        if not torch.isfinite(loss):
            print(f"[Skipping batch: loss non-finite @ step {step}]")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        running_loss += loss.item() * grad_accumulation_steps * src.size(0)
        total_examples += src.size(0)

        if step % grad_accumulation_steps == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.tensor(nn.utils.clip_grad_norm_(model.parameters(), grad_clip))
            if torch.isfinite(grad_norm):
                scaler.step(optimizer)
                scaler.update()
            else:
                print(f"[Skipping step: grad_norm non-finite @ step {step}]")
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                continue
            optimizer.zero_grad(set_to_none=True)

        # After optimizer step, check model parameters
        if model_has_nonfinite(model):
            print(f"Stopping: model weights became non-finite after step {step}.")
            break

        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'best_CER': f'{best_val_cer:.4f}'
        })

    if len(train_loader) % grad_accumulation_steps != 0:
        scaler.unscale_(optimizer)
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        if torch.isfinite(torch.tensor(grad_norm)):
            scaler.step(optimizer)
            scaler.update()
        else:
            print("[Final mini-batch grad_norm non-finite]")
            scaler.update()
        optimizer.zero_grad(set_to_none=True)

    progress_bar.close()

    scheduler.step()
    train_loss = running_loss / max(total_examples, 1)
    valid_loss, valid_cer, valid_exact = evaluate_model(model, dev_loader, criterion)
    history.append((train_loss, valid_loss, valid_cer, valid_exact))

    # Dynamically update exploded loss threshold to trip ~3x worst previous seen (for catastrophic detection)
    if len(history) > 3:
        exploded_loss_threshold = max(3.0, 3.0 * min([v[0] for v in history if torch.isfinite(torch.tensor(v[0]))]))

    print(
        f'Epoch {epoch:02d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_cer={valid_cer:.4f} | valid_exact={valid_exact:.4f} | noise_apply_prob={NOISE_STATE["apply_prob"]:.2f} | noise_heavy_prob={NOISE_STATE["heavy_prob"]:.2f}'
    )

    if not torch.isfinite(torch.tensor(train_loss)) or train_loss > exploded_loss_threshold:
        print(f"[Training loss became non-finite or exploded at epoch {epoch}]")
        break
    if not torch.isfinite(torch.tensor(valid_loss)) or not torch.isfinite(torch.tensor(valid_cer)):
        print(f"[Eval loss or CER non-finite.]")
        break

    if valid_cer < best_val_cer and torch.isfinite(torch.tensor(valid_loss)) and torch.isfinite(torch.tensor(valid_cer)):
        best_val_cer = valid_cer
        patience_counter = 0
        torch.save({
            'model_state': {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'optimizer_state': optimizer.state_dict(),
            'scheduler_state': scheduler.state_dict(),
            'scaler_state': scaler.state_dict(),
            'epoch': epoch,
            'best_val_cer': best_val_cer,
            'patience_counter': patience_counter,
            'history': history,
        }, checkpoint_path)
        print('Checkpoint saved.')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print('Early stopping triggered.')
            break

    # OPTIONAL: Print noisy examples every 5 epochs
    if epoch % 5 == 0:
        sample_batch = next(iter(train_loader))
        # print decoded src/tgt for sanity-check
        print("Sample noisy src_ids:", sample_batch['src'][0])
        print("Sample tgt_ids:", sample_batch['tgt'][0])

# --- After training, restore best weights ---
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    print(f'Loaded best model from epoch {checkpoint["epoch"]}, best CER: {checkpoint["best_val_cer"]:.4f}')

print(f'Best validation CER: {best_val_cer:.4f}')
print(f'Saved checkpoint: {checkpoint_path.resolve()}')

In [ ]:
def transliterate(text):
    model.eval()
    normalized = normalize_text(text).lower()
    src_ids = torch.tensor(text_to_ids(normalized, max_length=SRC_MAX_LEN), dtype=torch.long).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        output_ids = model(src_ids, trg=None, max_len=TGT_MAX_LEN)[0].tolist()
    return ids_to_text(output_ids)

sample_inputs = ['pariskrit', 'disha', 'gorkhalandka', 'chanchaltaharu', 'rukho']
for sample in sample_inputs:
    print(sample, '->', transliterate(sample))